# 02 — Lalonde Benchmark: Foundation Models vs. Metalearners

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chris-L6/causalfm-survey/blob/main/notebooks/02_lalonde_benchmark.ipynb)

**Compare one foundation model against traditional metalearners on the Lalonde dataset.**

This notebook runs:
- 1 foundation model (choose: CausalPFN / Do-PFN / CausalFM)
- 6 metalearners (S-learner, T-learner, X-learner, Debiased ML, IPW, DR)

on the Lalonde real-world causal inference benchmark and produces a comparison table.

## 1. Setup

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/chris-L6/causalfm-survey.git"
REPO_DIR = "causalfm-survey"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    sys.path.insert(0, REPO_DIR)
else:
    sys.path.insert(0, os.path.abspath(".."))

import causal_bench
print("causal_bench imported from:", causal_bench.__file__)

In [ ]:
!pip install -q econml causalml causalpfn torch
import ipywidgets as widgets
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Device: {device}")

## 2. Load Lalonde Dataset

In [ ]:
from causal_bench import load_lalonde, evaluate_cate

print("Loading Lalonde dataset...")
ds = load_lalonde()
print(f"  n={len(ds.Y)}, X.shape={ds.X.shape}, observed ATE={ds.ate:.3f}")

train_idx, test_idx = ds.train_test_split(0.7, seed=0)
X_train, X_test = ds.X[train_idx], ds.X[test_idx]
T_train, Y_train = ds.T[train_idx], ds.Y[train_idx]

print(f"  train: n={len(train_idx)}, test: n={len(test_idx)}")

## 3. Select Foundation Model

In [ ]:
from causal_bench import CausalPFNWrapper, DoPFNWrapper, CausalFMWrapper

FOUNDATION_MODELS = {
    "CausalPFN": CausalPFNWrapper,
    "Do-PFN": DoPFNWrapper,
    "CausalFM": CausalFMWrapper,
}

available_foundation = {name: cls for name, cls in FOUNDATION_MODELS.items() if cls.is_available()}
print(f"Available foundation models: {list(available_foundation.keys())}")

foundation_dropdown = widgets.Dropdown(options=available_foundation.keys(), description="Model:")
display(foundation_dropdown)

## 4. Run All Models

In [ ]:
from causal_bench import (
    SLearnerWrapper, TLearnerWrapper, XLearnerWrapper,
    DebiasedMLWrapper, IPWWrapper, DRWrapper
)

METALEARNERS = {
    "S-learner": SLearnerWrapper,
    "T-learner": TLearnerWrapper,
    "X-learner": XLearnerWrapper,
    "Debiased ML": DebiasedMLWrapper,
    "IPW": IPWWrapper,
    "DR (Doubly Robust)": DRWrapper,
}

results = []

# Run metalearners
print("="*70)
print("METALEARNERS")
print("="*70)
for name, model_cls in METALEARNERS.items():
    if not model_cls.is_available():
        print(f"  {name:25s}: SKIPPED (not available)")
        continue

    try:
        t0 = time.time()
        model = model_cls()
        model.fit(X_train, T_train, Y_train)
        tau_hat, lower, upper = model.predict(X_test)
        runtime = time.time() - t0
        ate_hat = float(np.mean(tau_hat))

        result = {
            "model": name,
            "ate_hat": ate_hat,
            "ate_true": ds.ate,
            "ate_abs_error": abs(ate_hat - ds.ate),
            "ate_rel_error": abs(ate_hat - ds.ate) / (abs(ds.ate) + 1e-8),
            "runtime_s": runtime,
        }
        results.append(result)
        print(f"  {name:25s}: ATE_error={result['ate_abs_error']:.4f}, runtime={runtime:.2f}s")
    except Exception as e:
        print(f"  {name:25s}: ERROR: {e}")

# Run foundation model
print("\n" + "="*70)
print("FOUNDATION MODEL")
print("="*70)
fm_name = foundation_dropdown.value
print(f"  {fm_name}...")

try:
    t0 = time.time()
    model_cls = FOUNDATION_MODELS[fm_name]

    if fm_name == "CausalFM":
        checkpoint_path = "CausalFM-toolkit/checkpoints/best_model.pth"
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"CausalFM checkpoint not found at {checkpoint_path}")
        model = model_cls(checkpoint_path=checkpoint_path, device=device)
    else:
        model = model_cls(device=device)

    model.fit(X_train, T_train, Y_train)
    tau_hat, lower, upper = model.predict(X_test)
    runtime = time.time() - t0
    ate_hat = float(np.mean(tau_hat))

    result = {
        "model": fm_name + " (Foundation)",
        "ate_hat": ate_hat,
        "ate_true": ds.ate,
        "ate_abs_error": abs(ate_hat - ds.ate),
        "ate_rel_error": abs(ate_hat - ds.ate) / (abs(ds.ate) + 1e-8),
        "runtime_s": runtime,
    }
    results.append(result)
    print(f"  {fm_name:25s}: ATE_error={result['ate_abs_error']:.4f}, runtime={runtime:.2f}s")
except Exception as e:
    print(f"  {fm_name:25s}: ERROR: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*70)

## 5. Results Table

In [ ]:
df = pd.DataFrame(results)
df_sorted = df.sort_values("ate_abs_error")

print("\nResults (sorted by ATE absolute error):")
print(df_sorted[["model", "ate_hat", "ate_true", "ate_abs_error", "ate_rel_error", "runtime_s"]].to_string(index=False))

df_sorted.to_csv("lalonde_benchmark.csv", index=False)
print("\nSaved to lalonde_benchmark.csv")

## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ATE error comparison
ax = axes[0]
df_plot = df_sorted.copy()
colors = ['#1f77b4' if 'Foundation' in name else '#ff7f0e' for name in df_plot['model']]
ax.barh(range(len(df_plot)), df_plot['ate_abs_error'], color=colors)
ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(df_plot['model'])
ax.set_xlabel('Absolute ATE Error')
ax.set_title('ATE Error: Foundation vs. Metalearners')
ax.axvline(ds.ate, color='red', linestyle='--', alpha=0.5, label='True ATE')

# Runtime comparison
ax = axes[1]
ax.barh(range(len(df_plot)), df_plot['runtime_s'], color=colors)
ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(df_plot['model'])
ax.set_xlabel('Runtime (seconds)')
ax.set_title('Runtime: Fit + Predict on Test Set')

plt.legend(['Foundation', 'Metalearner'], loc='lower right')
plt.tight_layout()
plt.savefig("lalonde_benchmark.png", dpi=150)
plt.show()
print("Saved to lalonde_benchmark.png")

## Interpretation

**ATE Error** (lower is better):
- Measures how well each model estimates the average treatment effect
- Foundation models learn from large training priors; metalearners fit to this specific data

**Runtime** (lower is better):
- Foundation models: fast (forward pass only, no retraining)
- Metalearners: slower (train separate models for each group)

**Real data**: No ground-truth CATE available, only observed ATE (simple difference in means).
Foundation models may have learned causal relationships from their training priors that help here.